# How the M5 retail project was built, step by step

**The question.** Where is a 10-store retailer losing sales? Two suspects: a forecast that is wrong where it matters
(the shelf), and shelves that go empty.

**How this notebook is organised.** It follows the project in the order it was built:

| Step | What | Tool |
|---|---|---|
| 1 | Get the public M5 data and the competition winner's forecast | git, unzip |
| 2 | Load the wide files and check them | pandas, NumPy |
| 3 | Model a star schema with tests | dbt on DuckDB |
| 4 | Answer the questions: revenue, forecast error, suspected stock-outs | NumPy + pandas + Altair |
| 5 | Reconcile with the dbt marts | pandas + DuckDB |
| 6 | Publish: web dashboard, Power BI | JSON export, PBIP |

The data are **real**: the M5 Forecasting dataset (Walmart, 2020), daily unit sales of 3,049 products in 10 stores in
California, Texas and Wisconsin from January 2011 to June 2016, 47 million item-store-days, with weekly prices.

**The charts are interactive**: brush a time range, click a state, a store or a cell to filter the linked chart.
They are built with [Altair](https://altair-viz.github.io/); open the
[interactive version](https://guilhermerigonato.com/5-retail-bi/notebook/) to use them (GitHub shows notebooks
without JavaScript).

**Run it:** `pip install pandas numpy altair duckdb dbt-duckdb`, then from the repository root
`bash scripts/get_data.sh` and `dbt build --profiles-dir .`, and run all cells (about 2 minutes, 4 GB of memory).

In [1]:
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd
import altair as alt

RAW = Path("../data/raw")
pd.set_option("display.float_format", "{:,.2f}".format)
BLUE, ORANGE, GREEN, GREY, MUTED, GRID = "#2a78d6", "#eb6834", "#1baf7a", "#c9c8c3", "#52514e", "#e4e3df"
STATES = {"CA": "California", "TX": "Texas", "WI": "Wisconsin"}

@alt.theme.register("portfolio", enable=True)
def portfolio_theme():
    axis = {"gridColor": GRID, "domainColor": MUTED, "tickColor": MUTED, "labelColor": MUTED,
            "titleColor": MUTED, "labelFontSize": 11, "titleFontSize": 11, "titleFontWeight": "normal"}
    return {"config": {"view": {"stroke": None}, "axis": axis, "legend": {"labelColor": MUTED, "titleColor": MUTED},
                       "title": {"anchor": "start", "fontSize": 13, "color": "#0b0b0b"},
                       "font": "system-ui, -apple-system, Segoe UI, sans-serif"}}

## Step 1. Get the data

[`scripts/get_data.sh`](../scripts/get_data.sh) clones a public mirror of the competition files and the published
submission of the winner (team YJ_STU), and unzips them into `data/raw/`:

| File | What | Shape |
|---|---|---|
| `sales_train_evaluation.csv` | Daily units per item and store, days 1–1941 | one row per item-store, one column per day |
| `sales_test_evaluation.csv` | The 28 out-of-sample days, 1942–1969 | same |
| `sell_prices.csv` | Price per item, store and Walmart week | long |
| `calendar.csv` | Day number → date, week, events, SNAP days | long |
| `0001 YJ_STU.csv` | The winning forecast for the 28 days | one row per item-store, columns F1–F28 |

M5 has no inventory and no cost data. That shapes the questions: margin cannot be measured, and empty shelves have to
be inferred from sales.

## Step 2. Load the wide files and check them

The sales files are *wide*: 30,490 item-store rows by 1,969 day columns. Unpivoting them gives 60 million rows, which
is what dbt does in DuckDB. In Python a wide file is an advantage: it loads straight into a NumPy matrix, and every
calculation below is a matrix operation.

In [2]:
cal = pd.read_csv(RAW / "calendar.csv", parse_dates=["date"])
train = pd.read_csv(RAW / "sales_train_evaluation.csv")
test = pd.read_csv(RAW / "sales_test_evaluation.csv")
ids = train[["item_id", "dept_id", "cat_id", "store_id", "state_id"]]
assert (test[["item_id", "store_id"]].values == ids[["item_id", "store_id"]].values).all(), "same row order"
day_cols = [c for c in train.columns if c.startswith("d_")] + [c for c in test.columns if c.startswith("d_")]
units = np.hstack([train.filter(like="d_").to_numpy(np.float32), test.filter(like="d_").to_numpy(np.float32)])
del train, test
prices = pd.read_csv(RAW / "sell_prices.csv")
print(f"units matrix: {units.shape[0]:,} item-stores × {units.shape[1]:,} days = {units.size / 1e6:,.1f}m item-store-days")
print(f"calendar: {cal['date'].min():%d %b %Y} to {cal['date'].max():%d %b %Y}; prices: {len(prices):,} rows")

units matrix: 30,490 item-stores × 1,969 days = 60.0m item-store-days
calendar: 29 Jan 2011 to 19 Jun 2016; prices: 6,841,121 rows


In [3]:
# Price matrix by Walmart week, then expanded to days through the calendar
weeks = np.sort(cal["wm_yr_wk"].unique())
row_of = pd.Series(np.arange(len(ids)), index=pd.MultiIndex.from_frame(ids[["item_id", "store_id"]]))
p_rows = row_of.reindex(pd.MultiIndex.from_frame(prices[["item_id", "store_id"]])).to_numpy()
all_prices_map = not np.isnan(p_rows.astype(float)).any()
p_rows = p_rows.astype(np.int64)
p_cols = np.searchsorted(weeks, prices["wm_yr_wk"].to_numpy())
price_wk = np.full((len(ids), len(weeks)), np.nan, dtype=np.float32)
price_wk[p_rows, p_cols] = prices["sell_price"].to_numpy()
day_week = np.searchsorted(weeks, cal["wm_yr_wk"].to_numpy())
price = price_wk[:, day_week]                         # item-store × day; NaN = not on sale that week

checks = {
    "one row per item-store": not ids.duplicated(["item_id", "store_id"]).any(),
    "1,969 days, one per calendar date": units.shape[1] == len(cal) == 1969,
    "no negative units": (units >= 0).all(),
    "every price maps to an item-store": all_prices_map,
    "no sales in a week without a price": not ((units > 0) & np.isnan(price)).any(),
}
assert all(checks.values()), checks
pd.Series(checks, name="passes").to_frame()

,passes
one row per item-store,True
"1,969 days, one per calendar date",True
no negative units,True
every price maps to an item-store,True
no sales in a week without a price,True


The last check matters: an item only has a price from the week it was first stocked, so a missing price means
*not on sale yet*, not *zero sales*. The models below only count days with a price.

## Step 3. Model a star schema in dbt

The SQL lives in [`models/`](../models). Staging unpivots the wide files with DuckDB's `UNPIVOT` and types the
calendar and prices; the marts are `fct_sales_daily` (item × store × day with units, price and revenue),
`fct_forecast_eval` (the 28 forecast days next to the actuals), `dim_item`, `dim_store`, `dim_date`, and the
question marts `mart_sales_weekly`, `mart_forecast_accuracy`, `mart_zero_sales_runs` and `mart_item_store_profile`
(ABC-XYZ classes and demand patterns). `dbt build` runs 36 nodes, including tests for keys, relationships, accepted
values and row counts, in about a minute.

## Step 4. Answer the questions

### Where does the revenue come from?

Revenue = units × the week's shelf price. **Drag across the small chart** to zoom the large one; **click a state**
in the legend to highlight it.

In [4]:
revenue = np.nan_to_num(units * price)                # item-store × day, $
cal["week_start"] = cal["date"] - pd.to_timedelta(cal["date"].dt.weekday, unit="D")
full = (cal["week_start"] > "2011-01-24") & (cal["week_start"] <= "2016-06-13")   # complete weeks only
week_idx, week_starts = pd.factorize(cal["week_start"])
by_state = {}
for st in STATES:
    rows = (ids["state_id"] == st).to_numpy()
    daily = revenue[rows].sum(axis=0, dtype=np.float64)
    by_state[STATES[st]] = np.bincount(week_idx, weights=daily)
weekly = pd.DataFrame(by_state, index=week_starts)[full.groupby(week_idx).all().to_numpy()]
weekly_long = weekly.rename_axis("week").reset_index().melt("week", var_name="state", value_name="revenue")

brush = alt.selection_interval(encodings=["x"])
state = alt.selection_point(fields=["state"], bind="legend")
color = alt.Color("state:N", scale=alt.Scale(range=[BLUE, ORANGE, GREEN]), title="State (click)")
big = alt.Chart(weekly_long).mark_line(strokeWidth=1.6).encode(
    x=alt.X("week:T", title=None, scale=alt.Scale(domain=brush)),
    y=alt.Y("revenue:Q", title="Weekly revenue ($)", axis=alt.Axis(format="~s")),
    color=color, opacity=alt.condition(state, alt.value(1), alt.value(0.12)),
    tooltip=["state", alt.Tooltip("week:T", title="Week of", format="%d %b %Y"), alt.Tooltip("revenue:Q", format=",.0f")],
).add_params(state).properties(width=600, height=230, title="Weekly revenue by state, 2011–2016")
small = alt.Chart(weekly_long).mark_area(opacity=0.5).encode(
    x=alt.X("week:T", title="Drag here to zoom"), y=alt.Y("sum(revenue):Q", title=None, axis=None), color=alt.value(GREY),
).add_params(brush).properties(width=600, height=50)
alt.vconcat(big, small)

alt.VConcatChart(...)

In [5]:
last52 = revenue[:, -364:].sum(dtype=np.float64); prior52 = revenue[:, -728:-364].sum(dtype=np.float64)
cat_share = pd.Series(revenue[:, -364:].sum(axis=1, dtype=np.float64), index=ids["cat_id"]).groupby(level=0).sum() / last52
print(f"Revenue, last 52 weeks: ${last52 / 1e6:,.1f}m ({last52 / prior52 - 1:+.1%} on the prior 52 weeks)")
cat_share.rename("share of revenue").to_frame().style.format("{:.0%}")

Revenue, last 52 weeks: $45.6m (+14.9% on the prior 52 weeks)


,share of revenue
cat_id,
FOODS,56%
HOBBIES,13%
HOUSEHOLD,31%


### Is the forecast good enough to stock the shelf?

The winning forecast covers 23 May to 19 June 2016. Its error depends on the level it is judged at:
**WAPE** = sum of absolute errors ÷ sum of actual units (MAPE is useless here because most item-days sell zero).

In [6]:
fc = pd.read_csv(RAW / "0001 YJ_STU.csv")
fc = fc[fc["id"].str.endswith("_evaluation")].set_index("id")
key = ids["item_id"] + "_" + ids["store_id"] + "_evaluation"
forecast = fc.loc[key, [f"F{i}" for i in range(1, 29)]].to_numpy(np.float64)
actual = units[:, -28:].astype(np.float64)

def wape(f, a): return np.abs(f - a).sum() / a.sum()
store_day = lambda m: pd.DataFrame(m).groupby(ids["store_id"].to_numpy()).sum().to_numpy()
levels = pd.Series({
    "item × store × day (the shelf)": wape(forecast, actual),
    "item × store × week": wape(forecast.reshape(len(ids), 4, 7).sum(2), actual.reshape(len(ids), 4, 7).sum(2)),
    "store × day": wape(store_day(forecast), store_day(actual)),
    "total company × day": wape(forecast.sum(0), actual.sum(0)),
}, name="WAPE")
print(f"Bias (forecast ÷ actual − 1): {forecast.sum() / actual.sum() - 1:+.1%}")
levels.to_frame().style.format("{:.1%}")

Bias (forecast ÷ actual − 1): +0.2%


,WAPE
item × store × day (the shelf),75.4%
item × store × week,39.5%
store × day,5.1%
total company × day,2.1%


The same forecast is 5% wrong for a store's day and 75% wrong for one item on one shelf. Replenishment works at the
shelf level, so it needs safety stock sized for that error. **Click a cell** of the heat map (store × category,
store-day WAPE) to see that store and category day by day.

In [7]:
cells = []
daily = []
for (store, cat), rows in ids.groupby(["store_id", "cat_id"]).groups.items():
    f, a = forecast[rows].sum(0), actual[rows].sum(0)
    cells.append(dict(store=store, category=cat.title(), wape=wape(f, a), bias=f.sum() / a.sum() - 1))
    daily += [dict(store=store, category=cat.title(), date=d, series=s, units=v)
              for d, fa, aa in zip(cal["date"].iloc[-28:], f, a) for s, v in (("Forecast", fa), ("Actual", aa))]
cells, daily = pd.DataFrame(cells), pd.DataFrame(daily)

cell = alt.selection_point(name="cell", fields=["store", "category"], on="click", value=[{"store": "CA_3", "category": "Hobbies"}])
heat = alt.Chart(cells).mark_rect(stroke="white", strokeWidth=2).encode(
    x=alt.X("category:N", title=None, sort=["Foods", "Household", "Hobbies"], axis=alt.Axis(labelAngle=0)),
    y=alt.Y("store:N", title=None),
    color=alt.Color("wape:Q", title="WAPE", scale=alt.Scale(scheme="blues"), legend=alt.Legend(format="%")),
    opacity=alt.condition(cell, alt.value(1), alt.value(0.55)),
    tooltip=["store", "category", alt.Tooltip("wape:Q", format=".1%"), alt.Tooltip("bias:Q", format="+.1%")],
).add_params(cell).properties(width=210, height=260, title="Store-day error (click a cell)")
text = alt.Chart(cells).mark_text(fontSize=10).encode(
    x=alt.X("category:N", sort=["Foods", "Household", "Hobbies"]), y="store:N", text=alt.Text("wape:Q", format=".0%"),
    color=alt.condition("datum.wape > 0.1", alt.value("white"), alt.value("#0b0b0b")))
pick = "cell.store ? cell.store[0] + ' · ' + cell.category[0] : ''"
lines = alt.Chart(daily).transform_filter(cell).mark_line(point=True).encode(
    x=alt.X("date:T", title=None, axis=alt.Axis(format="%d %b")),
    y=alt.Y("units:Q", title="Units per day", scale=alt.Scale(zero=False)),
    color=alt.Color("series:N", scale=alt.Scale(domain=["Actual", "Forecast"], range=[BLUE, ORANGE]), title=None, legend=alt.Legend(orient="top")),
    strokeDash=alt.StrokeDash("series:N", scale=alt.Scale(domain=["Actual", "Forecast"], range=[[1, 0], [5, 4]]), legend=None),
    tooltip=["series", alt.Tooltip("date:T", format="%a %d %b"), alt.Tooltip("units:Q", format=",.0f")],
).properties(width=400, height=260, title=alt.Title(alt.expr(f"'Actual and forecast · ' + ({pick})")))
alt.hconcat(heat + text, lines).resolve_scale(color="independent")

alt.HConcatChart(...)

In [8]:
hob = cells[cells["category"] == "Hobbies"]
print(f"Hobbies: highest error in {(cells.loc[cells.groupby('store')['wape'].idxmax(), 'category'] == 'Hobbies').sum()} of 10 stores, "
      f"under-forecast in {(hob['bias'] < 0).sum()} of 10; overall bias "
      f"{forecast[(ids['cat_id'] == 'HOBBIES').to_numpy()].sum() / actual[(ids['cat_id'] == 'HOBBIES').to_numpy()].sum() - 1:+.1%}")

Hobbies: highest error in 10 of 10 stores, under-forecast in 8 of 10; overall bias -4.4%


### How much is lost to empty shelves?

M5 has no stock data, so empty shelves are *inferred*: a run of 7 to 56 days in a row without a sale, for an item
that sells two or more units a day on average, is treated as a suspected stock-out (longer gaps look like a
delisting). Lost units = run length × the item's average daily units; lost revenue uses its average price. It is an
estimate, not a measured loss. The run detection is done on the matrix: for each row, mark zero days, then find where
each run starts and ends.

In [9]:
U, P = units[:, -365:], price[:, -365:]               # last 365 days
on_sale = ~np.isnan(P)
avg_units = np.where(on_sale, U, 0).sum(1) / on_sale.sum(1)
avg_price = np.nanmean(P, axis=1)
fast = avg_units >= 2

zero = (U == 0) & on_sale & fast[:, None]
pad = np.zeros((len(ids), 1), dtype=bool)
edges = np.diff(np.hstack([pad, zero, pad]).astype(np.int8), axis=1)
r_start, c_start = np.nonzero(edges == 1)
r_end, c_end = np.nonzero(edges == -1)                # nonzero() is row-major, so starts and ends pair up
length = c_end - c_start
keep = (length >= 7) & (length <= 56)
dates = cal["date"].iloc[-365:].to_numpy()
runs = pd.DataFrame({"item_id": ids["item_id"].to_numpy()[r_start[keep]], "store_id": ids["store_id"].to_numpy()[r_start[keep]],
                     "category": ids["cat_id"].to_numpy()[r_start[keep]], "run_start": dates[c_start[keep]], "run_days": length[keep]})
runs["lost_units"] = runs["run_days"] * avg_units[r_start[keep]]
runs["lost_revenue"] = runs["lost_units"] * avg_price[r_start[keep]]
print(f"{fast.sum():,} fast-moving item-stores; {len(runs):,} suspected stock-outs; "
      f"estimated ${runs['lost_revenue'].sum() / 1e6:,.2f}m lost ({runs['lost_revenue'].sum() / last52:.1%} of revenue)")
runs.groupby("category")["lost_revenue"].sum().div(runs["lost_revenue"].sum()).rename("share of lost revenue").to_frame().style.format("{:.0%}")

4,803 fast-moving item-stores; 4,353 suspected stock-outs; estimated $0.79m lost (1.7% of revenue)


,share of lost revenue
category,
FOODS,80%
HOBBIES,3%
HOUSEHOLD,18%


**Click a store** on the left to see its largest suspected stock-outs on the right.

In [10]:
store_lost = runs.groupby(["store_id", "category"])["lost_revenue"].sum().reset_index()
top = runs.sort_values("lost_revenue", ascending=False).groupby("store_id").head(10)
st = alt.selection_point(name="st", fields=["store_id"], on="click", value=[{"store_id": "CA_3"}])
cat_colors = alt.Scale(domain=["FOODS", "HOUSEHOLD", "HOBBIES"], range=[BLUE, ORANGE, GREEN])
left = alt.Chart(store_lost).mark_bar().encode(
    y=alt.Y("store_id:N", title=None, sort="-x"),
    x=alt.X("sum(lost_revenue):Q", title="Estimated lost revenue, last 365 days ($)", axis=alt.Axis(format="~s")),
    color=alt.Color("category:N", scale=cat_colors, title="Category"),
    opacity=alt.condition(st, alt.value(1), alt.value(0.35)),
    tooltip=["store_id", "category", alt.Tooltip("lost_revenue:Q", format=",.0f")],
).add_params(st).properties(width=300, height=250, title="Suspected stock-outs by store (click one)")
right = alt.Chart(top).transform_filter(st).mark_bar().encode(
    y=alt.Y("item_id:N", title=None, sort="-x"),
    x=alt.X("lost_revenue:Q", title="Estimated lost revenue ($)"),
    color=alt.Color("category:N", scale=cat_colors, legend=None),
    tooltip=["item_id", alt.Tooltip("run_start:T", format="%d %b %Y"), "run_days", alt.Tooltip("lost_units:Q", format=",.0f"),
             alt.Tooltip("lost_revenue:Q", format=",.0f")],
).properties(width=300, height=250, title=alt.Title(alt.expr("'Items with the largest estimated loss in ' + (st.store_id ? st.store_id[0] : '')")))
alt.hconcat(left, right)

alt.HConcatChart(...)

**Answer.** Food is about 56% of revenue but about 80% of the estimated loss to empty shelves, and one store (CA_3)
accounts for almost a quarter of it. Together with a forecast that is 75% wrong at the shelf, and Hobbies
under-forecast in most stores, the next step is safety stock by item and store, starting with fast-moving food in CA_3.

## Step 5. Reconcile with the dbt marts

NumPy on the wide matrix and SQL on the unpivoted facts are two independent routes to the same numbers.

In [11]:
db = duckdb.connect("../data/m5.duckdb", read_only=True)
m_week = db.sql("select sum(revenue_usd) r from mart_sales_weekly where week_start > date '2011-01-24' and week_start <= date '2016-06-13'").df()["r"][0]
m_runs = db.sql("select count(*) n, sum(est_lost_revenue_usd) v from mart_zero_sales_runs").df().iloc[0]
m_store_day = db.sql('''
    with sd as (select store_id, date_day, sum(forecast_units) f, sum(actual_units) a from fct_forecast_eval group by all)
    select sum(abs(f - a)) / sum(a) w from sd''').df()["w"][0]
m_item_day = db.sql("select sum(abs(forecast_units - actual_units)) / sum(actual_units) w from fct_forecast_eval").df()["w"][0]
db.close()

recon = pd.DataFrame([
    ["Weekly revenue, complete weeks ($)", weekly.to_numpy().sum(), m_week],
    ["WAPE, item × store × day", levels.iloc[0], m_item_day],
    ["WAPE, store × day", levels.iloc[2], m_store_day],
    ["Suspected stock-outs", len(runs), m_runs["n"]],
    ["Estimated lost revenue ($)", runs["lost_revenue"].sum(), m_runs["v"]],
], columns=["measure", "NumPy / pandas", "dbt"])
recon["match"] = np.isclose(recon["NumPy / pandas"].astype(float), recon["dbt"].astype(float), rtol=1e-5)
assert recon["match"].all(), recon
recon

,measure,NumPy / pandas,dbt,match
0,"Weekly revenue, complete weeks ($)","195,213,445.69","195,213,445.49",True
1,"WAPE, item × store × day",0.75,0.75,True
2,"WAPE, store × day",0.05,0.05,True
3,Suspected stock-outs,"4,353.00","4,353.00",True
4,Estimated lost revenue ($),"794,631.22","794,631.74",True


All five agree (to float32 rounding on the revenue sum).

## Step 6. Publish

* **Web dashboard** ([`web/`](../web), live at [guilhermerigonato.com/5-retail-bi](https://guilhermerigonato.com/5-retail-bi/)):
  [`scripts/export_web_data.py`](../scripts/export_web_data.py) writes JSON from the marts; clicking a state, a
  store, a cell of the error table or a stock-out row filters the other panels.
* **Power BI** ([`powerbi/`](../powerbi)): PBIP with a TMDL semantic model on Parquet exports of the marts, with
  pages for sales, forecast accuracy, inventory policy (ABC-XYZ) and stock-out priorities.

## Which library for interactive charts

| Library | Linked charts | Output | Best for |
|---|---|---|---|
| **Altair** (Vega-Lite), used here | Built in: selections shared across charts | Static HTML, no server | Reports that must open anywhere |
| **Plotly** | Per chart; linking across charts needs Dash | HTML; a Dash app for linking | Rich single charts, maps |
| **Bokeh** | Linked brushing through a shared data source | HTML, or a Bokeh server | Large scatter plots, custom tools |
| **HoloViews + Panel** | `link_selections` in one call; Datashader for millions of points | Notebook or Panel app | Big data, notebook-to-app |
| **Dash / Streamlit** | Any, through Python callbacks | Needs a server | Internal tools with live data |